# 🫁 DenseNet121 Pneumonia Detection — 3-Phase Training Pipeline
### Optimised for Google Colab Free Tier (T4 GPU, ~12h session)

**Training strategy:**
1. **Phase 1 · Pre-training** — CheXpert + NIH ChestX-ray14 (+optional MIMIC-CXR) · ~560K images
2. **Phase 2 · Fine-tuning** — RSNA Pneumonia Detection (bounding-box-supervised)
3. **Phase 3 · Final-tuning** — Kaggle Binary Chest X-Ray (clean, radiologist-confirmed labels)

**Colab-safety features:**
- Checkpoint saved to Google Drive after **every epoch** — safe to restart mid-training
- Automatic resume from last checkpoint on session restart
- CosineAnnealingLR + AdamW + AMP (mixed precision)
- WeightedRandomSampler + pos_weight BCEWithLogitsLoss for class-imbalance correction

---
**Steps before running:**
1. Upload your datasets to Google Drive (see Cell 3)
2. Set your Kaggle API key (see Cell 2)
3. Run cells top-to-bottom — each phase saves its own checkpoint

## ⚙️ Cell 1 — Install Dependencies

In [ ]:
# ── Install all required packages ───────────────────────────────────────────
# Run this cell once per Colab session (takes ~2 min on a fresh runtime)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pydicom opencv-python-headless albumentations scikit-learn tqdm matplotlib
!pip install -q torchinfo kaggle

import torch, torchvision, os, sys
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY ⚠️'}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB" if torch.cuda.is_available() else "")

## 📁 Cell 2 — Mount Google Drive & Configure Paths

**Before running this cell:**
- Upload your CheXpert zip to `MyDrive/pneumonia_project/datasets/`
- Download your Kaggle API token (`kaggle.json`) from kaggle.com → Account → API
- Upload `kaggle.json` to your Drive at `MyDrive/pneumonia_project/kaggle.json`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# ── Base paths (everything persists on Drive across sessions) ────────────────
DRIVE_BASE    = '/content/drive/MyDrive/pneumonia_project'
CKPT_DIR      = f'{DRIVE_BASE}/checkpoints'
WEIGHTS_DIR   = f'{DRIVE_BASE}/weights'
DATASET_DIR   = f'{DRIVE_BASE}/datasets'

# ── Local paths (fast SSD on Colab instance — NOT persistent) ────────────────
LOCAL_BASE     = '/content/data'
CHEXPERT_DIR   = f'{LOCAL_BASE}/chexpert'
NIH_DIR        = f'{LOCAL_BASE}/nih'
RSNA_DIR       = f'{LOCAL_BASE}/rsna'
KAGGLE_DIR     = f'{LOCAL_BASE}/kaggle'
MIMIC_DIR      = f'{LOCAL_BASE}/mimic'   # Only if you have PhysioNet access

for d in [CKPT_DIR, WEIGHTS_DIR, DATASET_DIR, LOCAL_BASE,
          CHEXPERT_DIR, NIH_DIR, RSNA_DIR, KAGGLE_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Kaggle API setup ─────────────────────────────────────────────────────────
KAGGLE_JSON_DRIVE = f'{DATASET_DIR}/kaggle.json'
if os.path.exists(KAGGLE_JSON_DRIVE):
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.copy(KAGGLE_JSON_DRIVE, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print("✓ Kaggle API key configured")
else:
    print("⚠️  kaggle.json not found — upload it to:", KAGGLE_JSON_DRIVE)

print("\nAll directories ready:")
print(f"  Drive checkpoints : {CKPT_DIR}")
print(f"  Drive weights     : {WEIGHTS_DIR}")
print(f"  Local data        : {LOCAL_BASE}")

## 📦 Cell 3 — Transfer Datasets from WSL to Google Drive

**Run these commands in your WSL terminal (NOT in Colab):**

```bash
# 1. Copy the CheXpert zip to a Google Drive folder you have synced locally
#    (Replace the path with your actual Google Drive sync path on Windows)
cp /home/anshu/minor_project/Chexpertdataset.zip "/mnt/c/Users/YOUR_USERNAME/Google Drive/pneumonia_project/datasets/"

# 2. If you don't have Google Drive for Desktop installed:
#    a. Install rclone in WSL: sudo apt install rclone
#    b. Configure: rclone config  (choose Google Drive)
#    c. Upload:
rclone copy /home/anshu/minor_project/Chexpertdataset.zip gdrive:pneumonia_project/datasets/ --progress

# 3. Also upload your kaggle.json to Drive:
rclone copy ~/.config/kaggle/kaggle.json gdrive:pneumonia_project/datasets/ --progress
```

**Once the zip is on Drive, run the cell below to extract it inside Colab:**

In [ ]:
# ── Extract CheXpert from Drive to fast local SSD ───────────────────────────
CHEXPERT_ZIP = f'{DATASET_DIR}/Chexpertdataset.zip'

if os.path.exists(CHEXPERT_ZIP):
    print("Extracting CheXpert (this takes ~5-8 min)...")
    !unzip -q "{CHEXPERT_ZIP}" -d "{CHEXPERT_DIR}"
    # After extraction, expected structure:
    # /content/data/chexpert/CheXpert-v1.0-small/train/  + train.csv + valid.csv
    print("✓ CheXpert extracted")
    !ls "{CHEXPERT_DIR}"
else:
    print(f"⚠️  CheXpert zip not found at {CHEXPERT_ZIP}")
    print("   Upload it to your Drive first (see instructions above)")

# ── Find the train.csv automatically ────────────────────────────────────────
import glob
csvs = glob.glob(f'{CHEXPERT_DIR}/**/train.csv', recursive=True)
if csvs:
    CHEXPERT_CSV = csvs[0]
    CHEXPERT_IMG_ROOT = os.path.dirname(CHEXPERT_CSV)
    print(f"✓ CheXpert CSV : {CHEXPERT_CSV}")
    print(f"✓ CheXpert root: {CHEXPERT_IMG_ROOT}")
    import pandas as pd
    df = pd.read_csv(CHEXPERT_CSV)
    print(f"  Rows: {len(df):,}")
else:
    print("⚠️  Could not find train.csv — check your extraction path")

## 📦 Cell 4 — Download NIH ChestX-ray14 Dataset

In [ ]:
# ── Download NIH ChestX-ray14 via Kaggle API ────────────────────────────────
# Dataset: https://www.kaggle.com/datasets/nih-chest-xrays/data
# Size: ~45 GB — download to Drive once, extract to Colab SSD per session

NIH_ZIP_ON_DRIVE = f'{DATASET_DIR}/nih-chest-xrays.zip'

if not os.path.exists(NIH_ZIP_ON_DRIVE):
    print("Downloading NIH ChestX-ray14 (~45GB) to Drive... this takes ~20-30 min")
    os.makedirs(DATASET_DIR, exist_ok=True)
    !kaggle datasets download -d nih-chest-xrays/data -p "{DATASET_DIR}" --unzip=False
    print("✓ NIH download complete")
else:
    print(f"✓ NIH zip already on Drive: {NIH_ZIP_ON_DRIVE}")

# Extract to fast local SSD
if not os.path.exists(f'{NIH_DIR}/images'):
    print("Extracting NIH to local SSD (~5 min)...")
    !unzip -q "{NIH_ZIP_ON_DRIVE}" -d "{NIH_DIR}"
    print("✓ NIH extracted")

# Verify
NIH_CSV = f'{NIH_DIR}/Data_Entry_2017.csv'
NIH_IMG_DIR = f'{NIH_DIR}/images'
if os.path.exists(NIH_CSV):
    import pandas as pd
    df_nih = pd.read_csv(NIH_CSV)
    pneum_count = df_nih['Finding Labels'].str.contains('Pneumonia').sum()
    print(f"✓ NIH CSV loaded: {len(df_nih):,} rows | Pneumonia cases: {pneum_count:,}")
else:
    print(f"⚠️  NIH CSV not found at {NIH_CSV} — check extraction")

# Set to None to skip NIH if not available
USE_NIH = os.path.exists(NIH_CSV)
print(f"\nUSE_NIH = {USE_NIH}")

## 📦 Cell 5 — Download RSNA Pneumonia Detection Dataset

In [ ]:
# ── Download RSNA Pneumonia Detection Challenge ──────────────────────────────
# Kaggle competition: rsna-pneumonia-detection-challenge
# Size: ~11 GB (DICOM files)

RSNA_ZIP_ON_DRIVE = f'{DATASET_DIR}/rsna-pneumonia-detection-challenge.zip'

if not os.path.exists(RSNA_ZIP_ON_DRIVE):
    print("Downloading RSNA dataset (~11GB) to Drive...")
    !kaggle competitions download -c rsna-pneumonia-detection-challenge -p "{DATASET_DIR}"
    print("✓ RSNA download complete")
else:
    print(f"✓ RSNA zip already on Drive")

# Extract to local SSD
if not os.path.exists(f'{RSNA_DIR}/stage_2_train_labels.csv'):
    print("Extracting RSNA...")
    !unzip -q "{RSNA_ZIP_ON_DRIVE}" -d "{RSNA_DIR}"
    print("✓ RSNA extracted")

# Convert DICOM → PNG (required because pydicom is slow at runtime)
RSNA_PNG_DIR = f'{RSNA_DIR}/train_png'
if not os.path.exists(RSNA_PNG_DIR) or len(os.listdir(RSNA_PNG_DIR)) < 100:
    print("Converting DICOM files to PNG (~10 min)...")
    os.makedirs(RSNA_PNG_DIR, exist_ok=True)
    import pydicom, cv2, numpy as np
    from tqdm import tqdm
    dicom_dir = f'{RSNA_DIR}/stage_2_train_images'
    dicoms = [f for f in os.listdir(dicom_dir) if f.endswith('.dcm')]
    for fname in tqdm(dicoms, desc="DICOM→PNG"):
        ds  = pydicom.dcmread(os.path.join(dicom_dir, fname))
        arr = ds.pixel_array.astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6) * 255
        arr = arr.astype(np.uint8)
        cv2.imwrite(os.path.join(RSNA_PNG_DIR, fname.replace('.dcm', '.png')), arr)
    print(f"✓ Converted {len(dicoms):,} DICOMs to PNG")
else:
    print(f"✓ RSNA PNGs already exist: {len(os.listdir(RSNA_PNG_DIR)):,} files")

RSNA_CSV = f'{RSNA_DIR}/stage_2_train_labels.csv'
RSNA_IMG_DIR = RSNA_PNG_DIR
USE_RSNA = os.path.exists(RSNA_CSV)
print(f"\nUSE_RSNA = {USE_RSNA}")

## 📦 Cell 6 — Download Kaggle Binary Chest X-Ray Dataset

In [ ]:
# ── Download Kaggle Chest X-Ray (Pneumonia) binary dataset ──────────────────
# Dataset: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
# Size: ~1.1 GB — very manageable

KAGGLE_ZIP_ON_DRIVE = f'{DATASET_DIR}/chest-xray-pneumonia.zip'

if not os.path.exists(KAGGLE_ZIP_ON_DRIVE):
    print("Downloading Kaggle binary dataset (~1.1 GB)...")
    !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p "{DATASET_DIR}"
    print("✓ Kaggle binary download complete")
else:
    print("✓ Kaggle binary zip already on Drive")

if not os.path.exists(f'{KAGGLE_DIR}/train/PNEUMONIA'):
    print("Extracting Kaggle binary dataset...")
    !unzip -q "{KAGGLE_ZIP_ON_DRIVE}" -d "{KAGGLE_DIR}"
    print("✓ Extracted")

# Detect folder structure
import os
for sub in ['chest_xray/train', 'train', '']:
    candidate = os.path.join(KAGGLE_DIR, sub)
    if os.path.isdir(os.path.join(candidate, 'PNEUMONIA')):
        KAGGLE_TRAIN_DIR = candidate
        break

for sub in ['chest_xray/val', 'val', 'chest_xray/test', 'test']:
    candidate = os.path.join(KAGGLE_DIR, sub)
    if os.path.isdir(os.path.join(candidate, 'PNEUMONIA')):
        KAGGLE_VAL_DIR = candidate
        break
else:
    KAGGLE_VAL_DIR = None

n_pneu = len(os.listdir(f'{KAGGLE_TRAIN_DIR}/PNEUMONIA'))
n_norm = len(os.listdir(f'{KAGGLE_TRAIN_DIR}/NORMAL'))
print(f"✓ Kaggle train — PNEUMONIA: {n_pneu:,} | NORMAL: {n_norm:,}")
print(f"  KAGGLE_TRAIN_DIR: {KAGGLE_TRAIN_DIR}")
USE_KAGGLE = True

## 🧠 Cell 7 — DenseNet121 Model Architecture

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training device: {DEVICE}")

def get_densenet121_model(num_classes=2, pretrained=True):
    """
    DenseNet121 modified for multi-label chest X-ray classification.

    Why DenseNet121?
    ├─ Reference architecture from Stanford CheXNet (Rajpurkar et al. 2017)
    ├─ Dense connections preserve fine-grained texture (lung opacity,
    │  consolidation) all the way to the classifier — ResNet loses this
    ├─ 7M params vs ResNet50's 25M → less overfitting on medical data
    └─ Used in every top-performing CheXpert leaderboard submission

    Output: raw logits (no sigmoid) — use BCEWithLogitsLoss during training,
            apply sigmoid at inference time for probabilities.
    """
    weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
    model   = models.densenet121(weights=weights)

    # Replace classifier: 1024 features → num_classes outputs
    num_ftrs = model.classifier.in_features   # 1024
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2),                    # Regularise — chest X-rays are noisy
        nn.Linear(num_ftrs, num_classes)
    )
    return model


# ── Quick sanity check ────────────────────────────────────────────────────────
model_test = get_densenet121_model(num_classes=2, pretrained=False)
dummy      = torch.randn(2, 3, 224, 224)
out        = model_test(dummy)
print(f"Output shape: {out.shape}")   # Expected: [2, 2]
print(f"GradCAM layer: model.features.denseblock4")

# Print parameter count
total_params = sum(p.numel() for p in model_test.parameters())
print(f"Total parameters: {total_params/1e6:.1f}M")
del model_test

## 🗂️ Cell 8 — Dataset Classes & Transforms

In [ ]:
import os, numpy as np, pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
from torchvision import transforms


# ── Transforms ───────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_transform():
    return transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def get_val_transform():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


# ── Dataset 1: CheXpert ───────────────────────────────────────────────────────
class ChexpertDataset(Dataset):
    """
    U-label policy:
      Pneumonia → U-Ones  (uncertain = positive: minority class, U-Zeros mislabels ~30%)
      Edema     → U-Zeros (uncertain = negative: already large class)
    """
    def __init__(self, csv_file, root_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.root_dir  = root_dir
        self.transform = transform
        self.target_cols = ['Pneumonia', 'Edema']
        self.df['Path'] = self.df['Path'].apply(
            lambda x: x.replace('CheXpert-v1.0-small/', '').replace('CheXpert-v1.0/', '')
        )
        self.df['Pneumonia'] = self.df['Pneumonia'].fillna(0).replace(-1, 1)   # U-Ones
        self.df['Edema']     = self.df['Edema'].fillna(0).replace(-1, 0)       # U-Zeros
        self.df[self.target_cols] = self.df[self.target_cols].astype(float)

    def __len__(self):  return len(self.df)

    def __getitem__(self, idx):
        p = os.path.join(self.root_dir, self.df.iloc[idx]['Path'])
        try: img = Image.open(p).convert('RGB')
        except: img = Image.new('RGB', (224, 224), (0, 0, 0))
        lbl = torch.tensor(
            self.df.iloc[idx][self.target_cols].values.astype(np.float32)
        )
        if self.transform: img = self.transform(img)
        return img, lbl, p


# ── Dataset 2: NIH ChestX-ray14 ─────────────────────────────────────────────
class NIHDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p   = os.path.join(self.img_dir, row['Image Index'])
        try: img = Image.open(p).convert('RGB')
        except: img = Image.new('RGB', (224, 224), (0, 0, 0))
        findings  = str(row['Finding Labels']).split('|')
        pneumonia = 1.0 if 'Pneumonia' in findings else 0.0
        edema     = 1.0 if 'Edema'     in findings else 0.0
        lbl = torch.tensor([pneumonia, edema], dtype=torch.float32)
        if self.transform: img = self.transform(img)
        return img, lbl, p


# ── Dataset 3: RSNA Pneumonia ────────────────────────────────────────────────
class RSNADataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        df_raw   = pd.read_csv(csv_file)
        self.df  = df_raw.groupby('patientId', as_index=False)['Target'].max()
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p   = None
        for ext in ('.png', '.jpg', '.jpeg'):
            candidate = os.path.join(self.img_dir, row['patientId'] + ext)
            if os.path.exists(candidate): p = candidate; break
        try: img = Image.open(p).convert('RGB')
        except: img = Image.new('RGB', (224, 224), (0, 0, 0)); p = p or ''
        lbl = torch.tensor([float(row['Target']), 0.0], dtype=torch.float32)
        if self.transform: img = self.transform(img)
        return img, lbl, p


# ── Dataset 4: Kaggle Binary ─────────────────────────────────────────────────
class KaggleBinaryDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples   = []
        for label_name, label_val in [('NORMAL', 0.0), ('PNEUMONIA', 1.0)]:
            folder = os.path.join(root_dir, label_name)
            if not os.path.isdir(folder): continue
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(folder, fname), label_val))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        p, pneu_lbl = self.samples[idx]
        try: img = Image.open(p).convert('RGB')
        except: img = Image.new('RGB', (224, 224), (0, 0, 0))
        lbl = torch.tensor([pneu_lbl, 0.0], dtype=torch.float32)
        if self.transform: img = self.transform(img)
        return img, lbl, p


# ── Pos-weight utility ────────────────────────────────────────────────────────
def compute_pos_weight(pneumonia_pos, edema_pos, total):
    n = max(total, 1)
    pw = torch.tensor([
        (n - max(pneumonia_pos, 1)) / max(pneumonia_pos, 1),
        (n - max(edema_pos, 1))     / max(edema_pos, 1),
    ], dtype=torch.float32)
    print(f"  pos_weight → Pneumonia: {pw[0]:.1f}  Edema: {pw[1]:.1f}")
    return pw


print("✓ All dataset classes defined")

## 🔧 Cell 9 — Training Utilities (Checkpoint, Train Loop, Validate)

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import matplotlib.pyplot as plt


# ── Checkpoint ────────────────────────────────────────────────────────────────
def save_checkpoint(state, drive_path):
    torch.save(state, drive_path)
    print(f"  ✓ Checkpoint → {drive_path}")


def load_checkpoint(drive_path, model, optimizer=None, scheduler=None):
    """Returns (start_epoch, best_val_loss). Safe to call even if file absent."""
    if not os.path.exists(drive_path):
        print(f"  No checkpoint at {drive_path} — starting fresh")
        return 0, float('inf')
    ckpt = torch.load(drive_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    if optimizer  and 'optimizer_state'  in ckpt: optimizer.load_state_dict(ckpt['optimizer_state'])
    if scheduler  and 'scheduler_state'  in ckpt: scheduler.load_state_dict(ckpt['scheduler_state'])
    ep   = ckpt.get('epoch', 0)
    bvl  = ckpt.get('best_val_loss', float('inf'))
    print(f"  ✓ Resumed from epoch {ep+1} (best val loss: {bvl:.4f})")
    return ep + 1, bvl


# ── Per-epoch train / validate ────────────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total = 0.0
    bar   = tqdm(loader, total=len(loader), leave=True)
    for imgs, lbls, _ in bar:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(imgs), lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
        bar.set_description(f"  loss {loss.item():.4f}")
    return total / len(loader)


def validate(model, loader, criterion):
    model.eval()
    total = 0.0
    bar   = tqdm(loader, total=len(loader), leave=True, desc="  val")
    with torch.no_grad():
        for imgs, lbls, _ in bar:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            with torch.amp.autocast('cuda'):
                total += criterion(model(imgs), lbls).item()
    return total / len(loader)


# ── Generic phase runner ──────────────────────────────────────────────────────
def run_phase(
    phase_name, model, train_loader, val_loader, pos_weight,
    epochs, lr, ckpt_path, best_path, freeze_backbone=False,
):
    print(f"\n{'─'*60}")
    print(f" {phase_name}")
    print(f"{'─'*60}")

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
    scaler    = torch.amp.GradScaler('cuda')

    if freeze_backbone:
        for p in model.features.parameters():
            p.requires_grad = False
        optimizer = optim.AdamW(model.classifier.parameters(), lr=lr, weight_decay=1e-4)
    else:
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    scheduler   = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr/100)
    start_epoch, best_val = load_checkpoint(ckpt_path, model, optimizer, scheduler)

    train_losses, val_losses = [], []

    for epoch in range(start_epoch, epochs):
        # After 1 warm-up epoch of fine-tuning: unfreeze backbone with differential LR
        if freeze_backbone and epoch == 1:
            print("  Unfreezing backbone with 10× lower LR")
            for p in model.features.parameters():
                p.requires_grad = True
            optimizer = optim.AdamW([
                {'params': model.features.parameters(),   'lr': lr / 10},
                {'params': model.classifier.parameters(),  'lr': lr},
            ], weight_decay=1e-4)
            scheduler = CosineAnnealingLR(optimizer, T_max=epochs - 1, eta_min=lr / 100)

        print(f"\n  Epoch {epoch+1}/{epochs}")
        tr_loss  = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_loss = validate(model, val_loader, criterion)
        scheduler.step()

        cur_lr = scheduler.get_last_lr()[0] if hasattr(scheduler.get_last_lr(), '__iter__') \
                 else scheduler.get_last_lr()
        print(f"  Train: {tr_loss:.4f}  Val: {val_loss:.4f}  LR: {cur_lr:.2e}")
        train_losses.append(tr_loss)
        val_losses.append(val_loss)

        # Save checkpoint to Drive every epoch (Colab-safe)
        save_checkpoint({
            'epoch':           epoch,
            'model_state':     model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_val_loss':   best_val,
        }, ckpt_path)

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), best_path)
            print(f"  ✓ New best weights → {best_path}")

    # Plot losses
    plt.figure(figsize=(8, 3))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses,   label='Val Loss')
    plt.title(phase_name); plt.xlabel('Epoch'); plt.legend(); plt.tight_layout()
    plt.savefig(best_path.replace('.pth', '_loss.png'))
    plt.show()

    print(f"\n  Phase complete. Best val loss: {best_val:.4f}")
    return model


print("✓ Training utilities ready")

## 🚀 Cell 10 — Phase 1: Pre-training on CheXpert + NIH ChestX-ray14

**Expected duration on T4 (Colab Free):**
- CheXpert only (~200K images, batch 64): ~2.5h per epoch
- CheXpert + NIH (~312K images, batch 64): ~4h per epoch
- **10 epochs ≈ 25–40h total** → run in 2–3 Colab sessions; checkpoint resumes automatically

**If Colab disconnects:** just re-run from Cell 1 to Cell 9, then re-run this cell — it resumes from the last completed epoch automatically.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PHASE 1 CONFIG — adjust to your dataset availability
# ════════════════════════════════════════════════════════════════════════════
P1_BATCH_SIZE = 64    # T4 handles 64 comfortably; raise to 96 if VRAM allows
P1_EPOCHS     = 10
P1_LR         = 1e-4
P1_NUM_WORKERS= 4

P1_CKPT_PATH  = f'{CKPT_DIR}/checkpoint_phase1.pth'
P1_BEST_PATH  = f'{WEIGHTS_DIR}/best_densenet121_phase1.pth'

# ════════════════════════════════════════════════════════════════════════════
#  BUILD DATALOADERS
# ════════════════════════════════════════════════════════════════════════════
# -- CheXpert split (90/10) --
chex_full  = ChexpertDataset(CHEXPERT_CSV, CHEXPERT_IMG_ROOT, transform=None)
n = len(chex_full)
tr_n, val_n = int(0.9*n), n - int(0.9*n)
gen = torch.Generator().manual_seed(42)
tr_idx, val_idx = torch.utils.data.random_split(range(n), [tr_n, val_n], generator=gen)

class _IndexedDataset(Dataset):
    """Wrap a base dataset with specific indices + a transform."""
    def __init__(self, base, indices, tf):
        self.base = base; self.indices = list(indices); self.tf = tf
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        real = self.indices[i]
        p = os.path.join(self.base.root_dir, self.base.df.iloc[real]['Path'])
        try: img = Image.open(p).convert('RGB')
        except: img = Image.new('RGB', (224, 224), (0,0,0))
        lbl = torch.tensor(self.base.df.iloc[real][self.base.target_cols].values.astype('float32'))
        return self.tf(img), lbl, p

train_tf, val_tf = get_train_transform(), get_val_transform()
train_datasets = [_IndexedDataset(chex_full, tr_idx,  train_tf)]
val_datasets   = [_IndexedDataset(chex_full, val_idx, val_tf)]

# Compute pos_weight from CheXpert CSV
df_tr = chex_full.df.iloc[list(tr_idx)]
pneu_pos = max(df_tr['Pneumonia'].sum(), 1)
edema_pos= max(df_tr['Edema'].sum(), 1)
pneu_tot = len(df_tr)

# -- NIH (optional) --
if USE_NIH and os.path.exists(NIH_CSV):
    nih_full = NIHDataset(NIH_CSV, NIH_IMG_DIR, transform=None)
    nih_n = len(nih_full)
    nih_tr_n = int(0.9*nih_n)
    nih_gen = torch.Generator().manual_seed(42)
    nih_tr, nih_val = torch.utils.data.random_split(range(nih_n), [nih_tr_n, nih_n-nih_tr_n], generator=nih_gen)

    class _NIHIndexed(Dataset):
        def __init__(self, base, indices, tf):
            self.base=base; self.indices=list(indices); self.tf=tf
        def __len__(self): return len(self.indices)
        def __getitem__(self, i):
            row = self.base.df.iloc[self.indices[i]]
            p   = os.path.join(self.base.img_dir, row['Image Index'])
            try: img = Image.open(p).convert('RGB')
            except: img = Image.new('RGB', (224,224),(0,0,0))
            fl  = str(row['Finding Labels']).split('|')
            lbl = torch.tensor([1.0 if 'Pneumonia' in fl else 0.0,
                                 1.0 if 'Edema'     in fl else 0.0], dtype=torch.float32)
            return self.tf(img), lbl, p

    train_datasets.append(_NIHIndexed(nih_full, nih_tr,  train_tf))
    val_datasets.append(  _NIHIndexed(nih_full, nih_val, val_tf))
    df_nih_tr = nih_full.df.iloc[list(nih_tr)]
    nih_fl = df_nih_tr['Finding Labels'].fillna('')
    pneu_pos  += nih_fl.str.contains('Pneumonia').sum()
    edema_pos += nih_fl.str.contains('Edema').sum()
    pneu_tot  += len(df_nih_tr)
    print(f"NIH added: {nih_n:,} images")

combined_train = ConcatDataset(train_datasets)
combined_val   = ConcatDataset(val_datasets)
print(f"Phase 1 — train: {len(combined_train):,} | val: {len(combined_val):,}")

# -- pos_weight --
p1_pos_weight = compute_pos_weight(pneu_pos, edema_pos, pneu_tot)

# -- WeightedRandomSampler (scan 30K sample subset for speed) --
_SCAN = min(30_000, len(combined_train))
_idx  = torch.randperm(len(combined_train))[:_SCAN].tolist()
_lbl  = np.array([combined_train[i][1].numpy() for i in _idx])
_n    = len(_lbl)
_cw_p = _n / max(_lbl[:,0].sum(), 1)
_cw_e = _n / max(_lbl[:,1].sum(), 1)
_w    = [max(1.0, (_cw_p if r[0]==1 else 0), (_cw_e if r[1]==1 else 0)) for r in _lbl]
p1_sampler = WeightedRandomSampler(torch.DoubleTensor(_w), _SCAN, replacement=True)

p1_train_loader = DataLoader(combined_train, batch_size=P1_BATCH_SIZE, sampler=p1_sampler,
                             num_workers=P1_NUM_WORKERS, drop_last=True, pin_memory=True)
p1_val_loader   = DataLoader(combined_val,   batch_size=P1_BATCH_SIZE, shuffle=False,
                             num_workers=P1_NUM_WORKERS, pin_memory=True)

# ════════════════════════════════════════════════════════════════════════════
#  RUN PHASE 1
# ════════════════════════════════════════════════════════════════════════════
p1_model = get_densenet121_model(num_classes=2, pretrained=True).to(DEVICE)

p1_model = run_phase(
    phase_name   = "Phase 1 — Pre-training (CheXpert + NIH)",
    model        = p1_model,
    train_loader = p1_train_loader,
    val_loader   = p1_val_loader,
    pos_weight   = p1_pos_weight,
    epochs       = P1_EPOCHS,
    lr           = P1_LR,
    ckpt_path    = P1_CKPT_PATH,
    best_path    = P1_BEST_PATH,
    freeze_backbone = False,
)
print("\n✅  Phase 1 complete — weights saved to Drive")

## 🔬 Cell 11 — Phase 2: Fine-tuning on RSNA Pneumonia Detection

**Expected duration on T4:** ~30K images, batch 64 → ~15 min/epoch → **5 epochs ≈ 1.5h**

Loads Phase 1 weights. Warm-up epoch with frozen backbone, then unfreezes with 10× lower LR on the backbone (differential learning rates).

In [ ]:
P2_BATCH_SIZE  = 64
P2_EPOCHS      = 5
P2_LR          = 1e-4
P2_NUM_WORKERS = 4

P2_CKPT_PATH   = f'{CKPT_DIR}/checkpoint_phase2.pth'
P2_BEST_PATH   = f'{WEIGHTS_DIR}/best_densenet121_phase2.pth'

if not USE_RSNA:
    print("⚠️  RSNA dataset not found — skipping Phase 2. Run Cell 5 first.")
else:
    rsna_full = RSNADataset(RSNA_CSV, RSNA_IMG_DIR, transform=None)
    n = len(rsna_full)
    tr_n, val_n = int(0.9*n), n - int(0.9*n)
    gen = torch.Generator().manual_seed(42)
    r_tr, r_val = torch.utils.data.random_split(range(n), [tr_n, val_n], generator=gen)

    class _RSNAIndexed(Dataset):
        def __init__(self, base, indices, tf):
            self.base=base; self.indices=list(indices); self.tf=tf
        def __len__(self): return len(self.indices)
        def __getitem__(self, i):
            row = self.base.df.iloc[self.indices[i]]
            p = None
            for ext in ('.png','.jpg','.jpeg'):
                candidate = os.path.join(self.base.img_dir, row['patientId']+ext)
                if os.path.exists(candidate): p=candidate; break
            try: img = Image.open(p).convert('RGB')
            except: img = Image.new('RGB',(224,224),(0,0,0)); p = p or ''
            lbl = torch.tensor([float(row['Target']), 0.0], dtype=torch.float32)
            return self.tf(img), lbl, p

    rsna_train_ds = _RSNAIndexed(rsna_full, r_tr,  get_train_transform())
    rsna_val_ds   = _RSNAIndexed(rsna_full, r_val, get_val_transform())

    # pos_weight for RSNA
    pneu_pos = max(rsna_full.df['Target'].sum(), 1)
    p2_pos_weight = compute_pos_weight(pneu_pos, 1, len(rsna_full))

    # Weighted sampler
    _cw  = len(rsna_full) / pneu_pos
    _w2  = [_cw if rsna_full.df.iloc[i]['Target']==1 else 1.0 for i in r_tr]
    p2_sampler = WeightedRandomSampler(torch.DoubleTensor(_w2), len(_w2), replacement=True)

    p2_train_loader = DataLoader(rsna_train_ds, batch_size=P2_BATCH_SIZE, sampler=p2_sampler,
                                 num_workers=P2_NUM_WORKERS, drop_last=True, pin_memory=True)
    p2_val_loader   = DataLoader(rsna_val_ds,   batch_size=P2_BATCH_SIZE, shuffle=False,
                                 num_workers=P2_NUM_WORKERS, pin_memory=True)
    print(f"RSNA — train: {len(rsna_train_ds):,} | val: {len(rsna_val_ds):,}")

    # Load Phase 1 weights
    p2_model = get_densenet121_model(num_classes=2, pretrained=False).to(DEVICE)
    if os.path.exists(P1_BEST_PATH):
        p2_model.load_state_dict(torch.load(P1_BEST_PATH, map_location=DEVICE))
        print(f"✓ Loaded Phase 1 weights: {P1_BEST_PATH}")
    else:
        print(f"⚠️  Phase 1 weights not found — training from ImageNet init")
        p2_model = get_densenet121_model(num_classes=2, pretrained=True).to(DEVICE)

    p2_model = run_phase(
        phase_name      = "Phase 2 — Fine-tuning (RSNA Pneumonia)",
        model           = p2_model,
        train_loader    = p2_train_loader,
        val_loader      = p2_val_loader,
        pos_weight      = p2_pos_weight,
        epochs          = P2_EPOCHS,
        lr              = P2_LR,
        ckpt_path       = P2_CKPT_PATH,
        best_path       = P2_BEST_PATH,
        freeze_backbone = True,
    )
    print("\n✅  Phase 2 complete — weights saved to Drive")

## 🎯 Cell 12 — Phase 3: Final Tuning on Kaggle Binary Pneumonia Dataset

**Expected duration on T4:** ~5,200 images, batch 64 → **~3 min/epoch → 3 epochs ≈ 10 min total**

This is the calibration step. The model already knows Pneumonia from Phase 1+2; this phase sharpens the decision boundary using radiologist-confirmed clean binary labels.

In [ ]:
P3_BATCH_SIZE  = 32    # Smaller — Kaggle dataset is small, avoid overfitting
P3_EPOCHS      = 5
P3_LR          = 5e-5  # Very low LR to preserve Phase 1+2 features
P3_NUM_WORKERS = 4

P3_CKPT_PATH  = f'{CKPT_DIR}/checkpoint_phase3.pth'
P3_BEST_PATH  = f'{WEIGHTS_DIR}/best_densenet121_FINAL.pth'   # ← Production weights

if not USE_KAGGLE:
    print("⚠️  Kaggle dataset not found — skip Phase 3. Run Cell 6 first.")
else:
    kaggle_train_ds = KaggleBinaryDataset(KAGGLE_TRAIN_DIR, transform=get_train_transform())
    if KAGGLE_VAL_DIR:
        kaggle_val_ds = KaggleBinaryDataset(KAGGLE_VAL_DIR, transform=get_val_transform())
    else:
        # 85 / 15 split from train dir
        all_kag = KaggleBinaryDataset(KAGGLE_TRAIN_DIR, transform=None)
        ktr_n = int(0.85 * len(all_kag))
        kval_n = len(all_kag) - ktr_n
        kgen  = torch.Generator().manual_seed(42)
        ktr, kval = torch.utils.data.random_split(all_kag, [ktr_n, kval_n], generator=kgen)

        class _KaggleSplit(Dataset):
            def __init__(self, sub, tf):
                self.sub=sub; self.tf=tf
            def __len__(self): return len(self.sub)
            def __getitem__(self, i):
                p, lv = self.sub.dataset.samples[self.sub.indices[i]]
                try: img = Image.open(p).convert('RGB')
                except: img = Image.new('RGB',(224,224),(0,0,0))
                return self.tf(img), torch.tensor([lv, 0.0], dtype=torch.float32), p

        kaggle_train_ds = _KaggleSplit(ktr,  get_train_transform())
        kaggle_val_ds   = _KaggleSplit(kval, get_val_transform())

    # pos_weight
    n_all_kag = KaggleBinaryDataset(KAGGLE_TRAIN_DIR)
    n_pneu = sum(1 for _, v in n_all_kag.samples if v == 1.0)
    n_total = len(n_all_kag.samples)
    p3_pos_weight = compute_pos_weight(n_pneu, 1, n_total)

    # Weighted sampler
    _cw3 = n_total / max(n_pneu, 1)
    _w3  = [_cw3 if s[1]==1.0 else 1.0 for s in n_all_kag.samples]
    p3_sampler = WeightedRandomSampler(torch.DoubleTensor(_w3), len(_w3), replacement=True)

    p3_train_loader = DataLoader(kaggle_train_ds, batch_size=P3_BATCH_SIZE, sampler=p3_sampler,
                                 num_workers=P3_NUM_WORKERS, drop_last=True, pin_memory=True)
    p3_val_loader   = DataLoader(kaggle_val_ds,   batch_size=P3_BATCH_SIZE, shuffle=False,
                                 num_workers=P3_NUM_WORKERS, pin_memory=True)
    print(f"Kaggle — train: {len(kaggle_train_ds):,} | val: {len(kaggle_val_ds):,}")

    # Load Phase 2 weights (or fall back to Phase 1)
    p3_model = get_densenet121_model(num_classes=2, pretrained=False).to(DEVICE)
    phase_src = P2_BEST_PATH if os.path.exists(P2_BEST_PATH) else P1_BEST_PATH
    if os.path.exists(phase_src):
        p3_model.load_state_dict(torch.load(phase_src, map_location=DEVICE))
        print(f"✓ Loaded weights: {phase_src}")
    else:
        print("⚠️  No Phase 1/2 weights — training from ImageNet init")
        p3_model = get_densenet121_model(num_classes=2, pretrained=True).to(DEVICE)

    p3_model = run_phase(
        phase_name      = "Phase 3 — Final Tuning (Kaggle Binary)",
        model           = p3_model,
        train_loader    = p3_train_loader,
        val_loader      = p3_val_loader,
        pos_weight      = p3_pos_weight,
        epochs          = P3_EPOCHS,
        lr              = P3_LR,
        ckpt_path       = P3_CKPT_PATH,
        best_path       = P3_BEST_PATH,
        freeze_backbone = True,
    )
    print(f"\n✅  ALL PHASES COMPLETE")
    print(f"   Final production weights: {P3_BEST_PATH}")

## 📊 Cell 13 — Evaluation & Metrics

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, roc_curve,
                              confusion_matrix, classification_report,
                              precision_recall_curve, average_precision_score)
import torch

# ── Load best final weights ────────────────────────────────────────────────
eval_model = get_densenet121_model(num_classes=2, pretrained=False).to(DEVICE)
if os.path.exists(P3_BEST_PATH):
    eval_model.load_state_dict(torch.load(P3_BEST_PATH, map_location=DEVICE))
    print(f"✓ Loaded: {P3_BEST_PATH}")
elif os.path.exists(P2_BEST_PATH):
    eval_model.load_state_dict(torch.load(P2_BEST_PATH, map_location=DEVICE))
    print(f"✓ Loaded (Phase 2 fallback): {P2_BEST_PATH}")
else:
    eval_model.load_state_dict(torch.load(P1_BEST_PATH, map_location=DEVICE))
    print(f"✓ Loaded (Phase 1 fallback): {P1_BEST_PATH}")
eval_model.eval()

# ── Run inference on Kaggle test set ─────────────────────────────────────────
eval_ds = KaggleBinaryDataset(KAGGLE_TRAIN_DIR, transform=get_val_transform()) \
          if KAGGLE_VAL_DIR is None else \
          KaggleBinaryDataset(KAGGLE_VAL_DIR, transform=get_val_transform())
eval_loader = DataLoader(eval_ds, batch_size=64, shuffle=False, num_workers=4)

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, lbls, _ in eval_loader:
        logits = eval_model(imgs.to(DEVICE))
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(lbls.numpy())

all_probs  = np.vstack(all_probs)
all_labels = np.vstack(all_labels)

# Pneumonia is class index 0
y_true = all_labels[:, 0].astype(int)
y_prob = all_probs[:,  0]
y_pred = (y_prob > 0.5).astype(int)

# ── Metrics ───────────────────────────────────────────────────────────────────
auc = roc_auc_score(y_true, y_prob)
print(f"\n{'─'*40}")
print(f"AUC (Pneumonia):   {auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Pneumonia']))

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0,1],[0,1],'k--'); axes[0].set_title('ROC Curve (Pneumonia)')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend()

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
im = axes[1].imshow(cm, cmap='Blues')
axes[1].set_title('Confusion Matrix')
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels(['Pred Normal','Pred Pneumonia'])
axes[1].set_yticklabels(['True Normal','True Pneumonia'])
for i in range(2):
    for j in range(2):
        axes[1].text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=14,
                     color='white' if cm[i,j]>cm.max()//2 else 'black')

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_true, y_prob)
ap = average_precision_score(y_true, y_prob)
axes[2].plot(recall, precision, color='darkorange', lw=2, label=f'AP = {ap:.3f}')
axes[2].set_title('Precision-Recall Curve'); axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision'); axes[2].legend()

plt.tight_layout()
plt.savefig(f'{WEIGHTS_DIR}/evaluation_metrics.png', dpi=150)
plt.show()
print(f"\n✓ Evaluation plots saved to Drive")

## 💾 Cell 14 — Export & Download Final Model

In [ ]:
import torch
from google.colab import files

# ── 1. The weights are already on Drive — just confirm ────────────────────────
final_weight = P3_BEST_PATH if os.path.exists(P3_BEST_PATH) else \
               P2_BEST_PATH if os.path.exists(P2_BEST_PATH) else P1_BEST_PATH
print(f"Final weights location on Drive:\n  {final_weight}")

# ── 2. Also export as TorchScript for production use ─────────────────────────
scripted_path = final_weight.replace('.pth', '_scripted.pt')
if os.path.exists(final_weight):
    export_model = get_densenet121_model(num_classes=2, pretrained=False).to('cpu')
    export_model.load_state_dict(torch.load(final_weight, map_location='cpu'))
    export_model.eval()
    try:
        scripted = torch.jit.script(export_model)
        scripted.save(scripted_path)
        print(f"✓ TorchScript saved: {scripted_path}")
    except Exception as e:
        print(f"TorchScript export failed ({e}) — using state_dict only")

# ── 3. Download directly to your PC ──────────────────────────────────────────
print("\nDownloading final weights to your computer...")
files.download(final_weight)

# ── 4. Instructions to copy from Drive to WSL ────────────────────────────────
print("""
─────────────────────────────────────────────────
COPY FROM GOOGLE DRIVE TO YOUR WSL PROJECT
─────────────────────────────────────────────────
Option A — rclone (recommended):
  rclone copy gdrive:pneumonia_project/weights/best_densenet121_FINAL.pth \\
              /home/anshu/minor_project/ --progress

Option B — Google Drive for Desktop on Windows:
  cp "/mnt/c/Users/<YOUR_USERNAME>/Google Drive/pneumonia_project/weights/best_densenet121_FINAL.pth" \\
     /home/anshu/minor_project/

Then update app.py weight_file to 'best_densenet121_FINAL.pth' — done!
─────────────────────────────────────────────────
""")